# Whisper large-v3-turbo — DIMER ASR tutorial

**Profile:** `TASK-INFERENCE`  
**Capability:** multilingual automatic speech recognition using the pinned OpenAI Whisper large-v3-turbo weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API rather than reimplementing model inference. The default sample is demonstration evidence, not a production-quality or benchmark claim.

**Learning objectives:** resolve the immutable upstream model revision, validate a public/default input, run the supported task, inspect task-appropriate outputs, exercise an optional BYOD path, and export machine-readable outputs plus provenance.


## Prerequisites

Run in a fresh supported runtime. Install dependencies before importing PyTorch or Transformers. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.


## 1. Bootstrap the repository and pinned runtime

The installation cell installs this repository and its exact model-facing dependency versions. If installation replaces a pre-imported core framework, restart the runtime before continuing.

In [ ]:
# Install before importing the model stack.
%pip install -q -e ".[tutorial]"
import platform, torch, transformers
print({'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Load a public sample or optional BYOD

The default path loads the public `hf-internal-testing/librispeech_asr_dummy` sample and its reference text. BYOD is optional and disabled by default; expected BYOD input is a local audio file readable by the ASR stack.

In [ ]:
from datasets import load_dataset
USE_BYOD = False
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    audio_input = next(iter(uploaded))
    reference = None
else:
    ds = load_dataset('hf-internal-testing/librispeech_asr_dummy', 'clean', split='validation')
    row = ds[0]
    audio_input = {'array': row['audio']['array'], 'sampling_rate': row['audio']['sampling_rate']}
    reference = row['text']
print({'sample_type': 'BYOD' if USE_BYOD else 'public LibriSpeech dummy', 'has_reference': reference is not None})

## 3. Resolve the pinned model

The public API pins the exact upstream revision and refuses remote model code. The effective model identity is printed before inference.

In [ ]:
from whisper_asr_pipeline import MODEL_ID, MODEL_REVISION, WhisperASRPipeline, word_error_rate
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})
pipe = WhisperASRPipeline.from_pretrained()
print({'device': pipe.device})

## 4. Run ASR and evaluate when ground truth exists

The pipeline returns normalized text plus provenance fields. WER is computed only when a reference transcript is available and is tutorial evidence, not a fleet benchmark.

In [ ]:
result = pipe.transcribe(audio_input, language='en', task='transcribe')
print(result['text'])
metrics = {}
if reference is not None:
    metrics['word_error_rate'] = word_error_rate(reference, result['text'])
print(metrics)

## 5. Export outputs and provenance

Machine-readable JSON preserves the transcript, measured tutorial metric, model identifier, immutable revision, and runtime identity.

In [ ]:
import json, os, platform
os.makedirs('outputs', exist_ok=True)
payload = {'prediction': result, 'metrics': metrics, 'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device}, 'sample': 'BYOD' if USE_BYOD else 'hf-internal-testing/librispeech_asr_dummy'}
with open('outputs/whisper_asr_result.json','w',encoding='utf-8') as f: json.dump(payload,f,indent=2,ensure_ascii=False)
print('outputs/whisper_asr_result.json')

## Interpretation and limits

The ASR text is a model-generated transcript. WER, when shown, is tied to the single demonstrated reference and must not be generalized to other languages, speakers, domains, or capture conditions. The pipeline provides no diarization, speaker identity, biometric inference, or calibrated transcript-confidence threshold.

Successful execution proves that this repository revision can acquire the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

## References

- Upstream model: https://huggingface.co/openai/whisper-large-v3-turbo
- Whisper paper: https://arxiv.org/abs/2212.04356
- Repository model card: `../MODEL_CARD.md`
